# Pravaala: silver annotation of GRETIL with ByT5-Sanskrit

Segments, lemmatises and tags the GRETIL units prepared by `scripts/prepare_gretil.py`,
using the fine-tuned multitask ByT5-Sanskrit model (Nehrdich, Hellwig & Keutzer 2024,
`chronbmm/sanskrit5-multitask`, mode `SLM` = segmentation + lemma + morphosyntax).

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. Upload the folder `data/silver/gretil/shards/` from the repo to Google Drive as
   `MyDrive/pravaala_silver/shards/` (91 files, one unit per line).

**Built for the free tier**: results go to `MyDrive/pravaala_silver/annotated/` every few
batches, and a restarted session skips everything already done. When a session ends,
reconnect and run all cells again.

**Afterwards**: download `annotated/`, put it in the repo as `data/silver/gretil/annotated/`,
then run `python scripts/build_silver_corpus.py --base all_nolex --profile np`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, time, glob, torch
BASE = '/content/drive/MyDrive/pravaala_silver'
IN, OUT = f'{BASE}/shards', f'{BASE}/annotated'
os.makedirs(OUT, exist_ok=True)
shards = sorted(glob.glob(f'{IN}/shard_*.txt'))
print(len(shards), 'shards;', 'GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4')

In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer
MODEL = 'chronbmm/sanskrit5-multitask'
tok = AutoTokenizer.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL).to('cuda').eval()
PREFIX, MAX_LEN = 'SLM ', 512          # the analyser's own task prefix and length cap

import copy
model16 = None        # fp16 copy, made only if needed (converting in place would lose precision)

def get_model(half):
    global model16
    if half and model16 is None:
        model16 = copy.deepcopy(model).half().eval()
    return model16 if half else model

def analyse(lines, half=False):
    m = get_model(half)
    enc = tok([PREFIX + l for l in lines], return_tensors='pt', padding=True,
              truncation=True, max_length=MAX_LEN).to('cuda')
    # output is ~2.5-3.5x the input in bytes (form_lemma_TAG per word); cap it so one
    # runaway output cannot keep the whole batch decoding up to 512 bytes
    cap = min(MAX_LEN, 4 * max(len(l.encode()) for l in lines) + 16)
    with torch.no_grad():
        out = m.generate(**enc, max_length=cap, num_beams=1)
    return tok.batch_decode(out, skip_special_tokens=True)

## 1. Timing and precision check (run once)
Times fp32 against fp16 on 256 units of the first shard, and counts how often they disagree.
T5 models can overflow in fp16; use fp16 below only if it agrees with fp32 on (almost) all units.

In [ ]:
sample = sorted(open(shards[0], encoding='utf-8').read().split('\n')[:512], key=len)
res = {}
for half in (False, True):
    t = time.time(); outs = []
    for i in range(0, len(sample), 128):
        outs += analyse(sample[i:i+128], half=half)
    res[half] = (outs, time.time() - t)
same = sum(a == b for a, b in zip(res[False][0], res[True][0]))
total = sum(sum(1 for _ in open(s, encoding='utf-8')) for s in shards)
for half, (outs, sec) in res.items():
    print(f"{'fp16' if half else 'fp32'}: {len(sample)/sec:.1f} units/s -> ~{total/(len(sample)/sec)/3600:.1f} GPU-hours for all {total:,} units")
print(f'fp16 identical to fp32 on {same}/{len(sample)} units')
print('example:', sample[0], '\n ->', res[False][0][0])

## 2. Annotate (resumable)
Set `HALF` from the check above. A progress bar shows units/s and the ETA for everything;
stopping the cell at any time is safe (at most 5 batches are redone). Units are sorted by length inside each shard (less padding);
every output line is `index-in-shard <TAB> raw model output`, so order does not matter.

In [ ]:
HALF = False          # True only if the check showed fp16 == fp32
BATCH = 128           # halved automatically on out-of-memory
FLUSH_EVERY = 5       # batches between writes to Drive
import random
from tqdm.auto import tqdm
if not HALF:
    model16 = None; torch.cuda.empty_cache()

# Shards follow GRETIL's folder order (Veda, epic, ...). A fixed shuffled order means
# that if we stop early, the annotated part still mixes all genres.
order = shards[:]; random.Random(0).shuffle(order)
total = sum(sum(1 for _ in open(s, encoding='utf-8')) for s in shards)
def n_done():
    return sum(len({l.split('\t', 1)[0] for l in open(f, encoding='utf-8') if '\t' in l})
               for f in glob.glob(f'{OUT}/shard_*.tsv'))
done_all = n_done(); t0 = time.time(); new = 0
print(f'{done_all:,}/{total:,} units already done')

for s in order:
    name = os.path.basename(s).replace('.txt', '.tsv')
    lines = open(s, encoding='utf-8').read().split('\n')
    if lines and lines[-1] == '':
        lines = lines[:-1]
    outp = f'{OUT}/{name}'
    done = set()
    if os.path.exists(outp):
        done = {int(l.split('\t', 1)[0]) for l in open(outp, encoding='utf-8') if '\t' in l}
    todo = sorted((i for i in range(len(lines)) if i not in done), key=lambda i: len(lines[i]))
    if not todo:
        continue
    bar = tqdm(total=len(todo), desc=name, unit='unit')
    buf, b, k = [], 0, 0
    while b < len(todo):
        idx = todo[b:b+BATCH]
        try:
            outs = analyse([lines[i] for i in idx], half=HALF)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); BATCH = max(8, BATCH // 2); print('OOM -> batch', BATCH); continue
        buf += [f'{i}\t{o}\n' for i, o in zip(idx, outs)]
        b += len(idx); k += 1; new += len(idx); bar.update(len(idx))
        if k % FLUSH_EVERY == 0 or b >= len(todo):
            with open(outp, 'a', encoding='utf-8') as f:
                f.writelines(buf)
            buf = []
            rate = new / (time.time() - t0)
            bar.set_postfix_str(f'overall {done_all+new:,}/{total:,}, ETA {(total-done_all-new)/rate/3600:.0f} h')
    bar.close()
print('all shards done')


## 3. Progress / packaging

In [ ]:
n_in = sum(sum(1 for _ in open(s, encoding='utf-8')) for s in shards)
n_out = sum(len({l.split('\t',1)[0] for l in open(f, encoding='utf-8') if '\t' in l})
            for f in glob.glob(f'{OUT}/shard_*.tsv'))
print(f'{n_out:,} / {n_in:,} units annotated ({100*n_out/n_in:.1f}%)')
# when complete: zip for download
# !cd {BASE} && zip -qr annotated.zip annotated && ls -la annotated.zip